[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/en_clase/S05_06_crud_python_en_clase.ipynb)

# S05_06 · CRUD desde Python: leer y cargar datos en una base (en clase)

<h2 align="center">Que Python lea y escriba en la base de datos de TurisData</h2>

| | |
|---|---|
| ⏱ **Cuándo** | Jueves · unos 35 min |
| 👥 **Cómo** | En parejas: una persona escribe, la otra lee y comprueba (cambiad de papel al empezar la parte B) |
| 🧰 **Usáis** | 🌐 Google Colab (este cuaderno) · no hay que instalar nada |
| 🏁 **Al terminar** | Ejercicio 1 (leer) y ejercicio 3 (poblar la base con la serie diaria) con todo ✅ en el paso 5 |

**Este cuaderno tiene dos partes:**

- **1 · Guía paso a paso**: solo lo que hay que hacer. Va en dos bloques: **A · Leer** y **B · Crear**.
- **2 · Explicaciones**: el porqué de cada cosa, para cuando lo necesitéis (está al final, también en dos bloques: A y B).
- Si queréis, seguid solo la parte 1.

**Solo trae los dos ejercicios de clase**: el **1 trata de leer** de la base (parte A) y el **3 de crear**: meter datos en ella (parte B). Los demás (2, 4, 5, 6 y el mini-reto) están en el cuaderno completo [`S05_06_crud_python`](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/S05_06_crud_python.ipynb), para casa.

**El ejercicio 3 es también el paso 1 del reto del sprint**: cargar los datos en una base.

## Cómo se usa este cuaderno (leedlo una vez)

**Un cuaderno es una mezcla de textos y celdas de código.** Las celdas de código tienen a la izquierda un botón **▶**.

1. **Ejecutar una celda**
   - Pulsad el botón **▶** de la izquierda, o haced clic dentro de la celda y pulsad **Mayús + Intro**.
   - Debajo de la celda aparece **lo que ha salido**.
2. **De arriba abajo, en orden**
   - Cada celda usa lo que han hecho las anteriores.
   - Si os saltáis una, saldrá un error rojo que dice `NameError`. No pasa nada: ejecutad las que faltan y repetid.
3. **Las celdas con `...`**
   - Son los ejercicios. Los **tres puntos `...` se sustituyen** por vuestro código.
   - No borréis el resto de la celda.
4. **Cada paso termina en ✅**
   - Es lo que **tiene que salir**.
   - Los decimales lejanos pueden cambiar un poco en vuestro ordenador; lo demás tiene que ser **igual**.
   - Si no sale, no sigáis: mirad la parte 2, apartado «Si os atascáis».
5. **Si Colab se reinicia o cerráis la pestaña**
   - Se pierde lo ejecutado, no lo escrito.
   - Volved a ejecutar las celdas **desde el paso 0 hasta donde estabais**.

# 1 · Guía paso a paso

**Qué vais a hacer**

- **Parte A · Leer (ejercicio 1):** abrir la base de datos de TurisData desde Python y sacar **cuántas reservas hay** y la **tasa de cancelación de cada isla**.
- **Parte B · Crear (ejercicio 3):** guardar **dentro de la base** la serie diaria de ocupación (1096 días), que viene de un fichero CSV.
- Es un programa que lee y escribe en la base **sin abrir ninguna herramienta gráfica**.
- Trabajáis sobre una **copia** de la base: podéis equivocaros sin miedo.

## Antes de empezar · Paso 0 · Preparar el cuaderno (3 min)

| # | Qué hacéis |
|---|---|
| **0.1** | Ejecutad la celda de abajo (la de «Preparación»). Tarda unos segundos: busca los ficheros de datos y, si no están, los descarga. |
| **0.2** | Ejecutad la celda siguiente (las librerías y la **copia de la base**). |

✅ Con la primera sale `Preparación lista ✅` y con la segunda `Base de trabajo lista ✅`.

*No hace falta entender estas dos celdas: dejan preparado lo que usaremos después.*

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

RESULTADOS = {}

def comprobar(nombre, condicion, ayuda):
    """Escribe ✅ si la condición es cierta y ❌ con una pista si no. No hace falta entender cómo funciona."""
    try:
        ok = bool(condicion())
    except Exception:
        ok = False
    RESULTADOS[nombre] = ok
    print(("✅ " + nombre) if ok else ("❌ " + nombre + " → " + ayuda))

dato("turisdata.db")
dato("ocupacion_diaria.csv")
print("Preparación lista ✅")

In [ ]:
import shutil
import sqlite3
import tempfile

import pandas as pd

# Copia de trabajo de la base de datos (así podéis equivocaros sin miedo)
carpeta = tempfile.mkdtemp()
ruta_bd = os.path.join(carpeta, "turisdata_copia.db")
shutil.copy(dato("turisdata.db"), ruta_bd)
print("Base de trabajo lista ✅")

# Parte A · Leer (ejercicio 1) · unos 15 min

**Qué es CRUD y qué vais a hacer**

- **CRUD** son las cuatro cosas que se hacen con los datos de una base: **C**rear, **R**eer (leer), **U**pdate (actualizar) y **D**elete (borrar).
- En esta parte hacéis la **R: leer**. En la parte B, la **C: crear**.
- Vais a **conectar** Python con la base, hacer preguntas en SQL y recibir las respuestas en Python.
- Después, el **ejercicio 1**: contar las reservas y calcular la tasa de cancelación de cada isla.

## Paso 1 · Conectar con la base y mirarla (5 min)

**La base de datos**

- Es la de TurisData, con la que trabajasteis en el Sprint 3.
- Tiene varias **tablas**; hoy usamos dos:

| Tabla | Qué contiene | Columnas que usaremos |
|---|---|---|
| **`reservas`** | Una fila por reserva (6000) | `id_alojamiento`, `canal`, `cancelada` (1 = sí, 0 = no) |
| **`alojamientos`** | Una fila por alojamiento (20) | `id_alojamiento`, `nombre`, `isla` |

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **1.1** | Ejecutad la celda 1.1 (abre la conexión `con` y cuenta las reservas). | `(6000,)` |
| **1.2** | Ejecutad la celda 1.2 (pide 3 alojamientos). | `[(1, 'Alojamiento 01', 'Tenerife'), (2, 'Alojamiento 02', 'Gran Canaria'), (3, 'Alojamiento 03', 'Lanzarote')]` |
| **1.3** | Ejecutad la celda 1.3 (reservas por canal, como una tabla de pandas). | Una tabla con 4 canales: `agencia 1190`, `directo 1462`, `empresa 328` y `web_intermediaria 3020` |

- `con` es **la conexión**: el «cable» entre Python y la base. Todas las preguntas pasan por él.
- Hay **dos formas de recibir** la respuesta: como lista de tuplas (1.1 y 1.2) o como tabla de pandas (1.3).

In [ ]:
# 1.1 · Abrir la conexión y contar las reservas
con = sqlite3.connect(ruta_bd)
print(con.execute("SELECT COUNT(*) FROM reservas").fetchone())

In [ ]:
# 1.2 · Pedir tres alojamientos
print(con.execute("SELECT id_alojamiento, nombre, isla FROM alojamientos LIMIT 3").fetchall())

In [ ]:
# 1.3 · Lo mismo, pero la respuesta llega como una tabla de pandas
pd.read_sql_query("SELECT canal, COUNT(*) AS reservas FROM reservas GROUP BY canal", con)

## Paso 2 · Ejercicio 1 · Leer con pandas (10 min) 🏫

**Objetivo**

- Guardar en `n_reservas` **cuántas reservas hay**.
- Guardar en `tasa_por_isla` una **tabla de pandas** con dos columnas: `isla` y `pct_cancelacion` (el % de reservas canceladas de cada isla).
- La consulta SQL **ya está escrita**: es la que aprendisteis en el Sprint 3 (`JOIN` y `GROUP BY`). Vuestro trabajo es **lanzarla desde Python**.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **2.1** | **A mano, con todas las reservas.** Ejecutad la celda 2.1: la tasa de cancelación global. | Una tabla con `pct_cancelacion 45.9` |
| **2.2** | **Vuestro turno.** En la celda 2.2, sustituid los dos `...`: uno cuenta las reservas y el otro lanza la consulta con pandas. | No sale nada |
| **2.3** | Ejecutad la celda 2.3 (la comprobación). | Cuatro líneas con ✅ |
| **2.4** | Ejecutad la celda 2.4 (muestra los dos resultados). | `6000` y una tabla de 6 islas, de Fuerteventura (`45.6`) a Tenerife (`46.0`) |

- **Pistas para 2.2:**
  - `n_reservas = con.execute("SELECT COUNT(*) FROM reservas").fetchone()[0]` (como la 1.1, pero con `[0]` para quedarse solo con el número)
  - `tasa_por_isla = pd.read_sql_query(consulta, con)` (como la 1.3, pero con la `consulta` de arriba)
- Si sale ❌, leed la pista que aparece tras la flecha `→` y corregid.

In [ ]:
# 2.1 · A mano: la tasa de cancelación de todas las reservas (el 45.9 % se cancela)
pd.read_sql_query("SELECT ROUND(100.0 * AVG(cancelada), 1) AS pct_cancelacion FROM reservas", con)

In [ ]:
# 2.2 · TU TURNO: contar las reservas y lanzar la consulta de la tasa por isla
n_reservas = ...            # como la celda 1.1, pero con [0] al final: con.execute("SELECT COUNT(*) FROM reservas").fetchone()[0]

consulta = """
SELECT a.isla,
       ROUND(100.0 * AVG(r.cancelada), 1) AS pct_cancelacion
FROM reservas AS r
JOIN alojamientos AS a ON r.id_alojamiento = a.id_alojamiento
GROUP BY a.isla
ORDER BY a.isla
"""
tasa_por_isla = ...         # como la celda 1.3, pero con la consulta de arriba: pd.read_sql_query(consulta, con)

In [ ]:
# 2.3 · Comprobación del ejercicio 1 (no hay que cambiar nada)
comprobar("leer: n_reservas", lambda: n_reservas == 6000, "debe salir 6000: con.execute(\"SELECT COUNT(*) FROM reservas\").fetchone()[0]. ¿Has cambiado los puntos ... por tu código?")
comprobar("leer: columnas", lambda: list(tasa_por_isla.columns) == ["isla", "pct_cancelacion"], "las columnas deben ser isla y pct_cancelacion: pd.read_sql_query(consulta, con)")
comprobar("leer: una fila por isla", lambda: len(tasa_por_isla) == 6, "debe haber 6 filas, una por isla: usa la consulta tal cual, sin cambiarla")
comprobar("leer: tasa de Tenerife", lambda: tasa_por_isla.set_index("isla").loc["Tenerife", "pct_cancelacion"] == 46.0, "Tenerife debe salir 46.0: ¿lanzas la consulta con pd.read_sql_query(consulta, con)?")

In [ ]:
# 2.4 · Mirar el resultado
print(n_reservas)
print(tasa_por_isla)

✅ Tiene que salir: `✅ leer: n_reservas`, `✅ leer: columnas`, `✅ leer: una fila por isla` y `✅ leer: tasa de Tenerife`. Si falta alguno, no sigáis.

# Parte B · Crear (ejercicio 3) · unos 15 min

*Cambiad de papel: quien escribía, ahora lee y comprueba.*

**Qué es crear y qué vais a hacer**

- **Crear** es meter datos nuevos en la base: una fila suelta (`INSERT`) o una **tabla entera** desde un fichero.
- Primero **veréis cómo se inserta una fila** y por qué hace falta **confirmar** el cambio (paso 3).
- Después, el **ejercicio 3**: leer el CSV de la ocupación diaria y **guardarlo en la base** como una tabla nueva (paso 4).
- Para esta parte se usa la conexión `con` de la parte A: si no la tenéis, volved al paso 1.

## Paso 3 · Insertar una fila y confirmarla (5 min)

**Objetivo**

- Ver que los cambios en la base **no son definitivos hasta confirmarlos** con `commit`.
- Ver que `rollback` los deshace.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **3.1** | Ejecutad la celda 3.1: inserta un alojamiento nuevo, lo **confirma** y lo lee. | `[(21, 'Alojamiento 21', 'El Hierro', 'casa rural', 12)]` |
| **3.2** | Ejecutad la celda 3.2: inserta otro alojamiento **sin confirmar** y después lo **deshace**. | `Con la fila nueva: 22` y `Tras el rollback: 21` |

- `INSERT INTO … VALUES (?, ?, …)` usa **marcadores `?`** y los valores van en una tupla aparte: así se evitan errores y ataques.
- **`commit`:** «guardar» los cambios de verdad.
- **`rollback`:** «deshacer» lo que no se ha guardado.

In [ ]:
# 3.1 · Insertar un alojamiento y confirmar el cambio
con.execute("INSERT INTO alojamientos (id_alojamiento, nombre, isla, categoria, plazas) VALUES (?, ?, ?, ?, ?)",
            (21, "Alojamiento 21", "El Hierro", "casa rural", 12))
con.commit()
print(con.execute("SELECT * FROM alojamientos WHERE id_alojamiento = 21").fetchall())

In [ ]:
# 3.2 · Insertar SIN confirmar y deshacer
con.execute("INSERT INTO alojamientos (id_alojamiento, nombre, isla, categoria, plazas) VALUES (?, ?, ?, ?, ?)",
            (22, "Alojamiento 22", "La Graciosa", "apartamento", 6))
print("Con la fila nueva:", con.execute("SELECT COUNT(*) FROM alojamientos").fetchone()[0])
con.rollback()
print("Tras el rollback:", con.execute("SELECT COUNT(*) FROM alojamientos").fetchone()[0])

## Paso 4 · Ejercicio 3 · Poblar la base con la serie diaria (10 min) 🏫

**Objetivo**

- Leer `ocupacion_diaria.csv` con pandas y guardar la tabla entera **dentro de la base**, con el nombre `ocupacion_diaria`.
- Comprobar leyendo **desde la base** que han entrado los 1096 días.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **4.1** | **A mano, con una tabla pequeña.** Ejecutad la celda 4.1: guarda una tabla de 2 filas en la base con `to_sql`. | `Filas en tabla_prueba: 2` |
| **4.2** | **Vuestro turno.** En la celda 4.2, sustituid los **tres `...`**: leer el CSV, guardarlo en la base y contar las filas **desde la base**. | No sale nada |
| **4.3** | Ejecutad la celda 4.3 (la comprobación). | Tres líneas con ✅ |
| **4.4** | Ejecutad la celda 4.4 (lee 3 filas de la nueva tabla, desde la base). | Una tabla de 3 filas con `fecha`, `ocupacion_pct`, `temperatura_c`, `festivo` y `precio_medio`; la primera es `2023-01-01` |

- **Pistas para 4.2:**
  - `diaria = pd.read_csv(dato("ocupacion_diaria.csv"))`
  - `diaria.to_sql("ocupacion_diaria", con, if_exists="replace", index=False)`
  - `n_dias = con.execute("SELECT COUNT(*) FROM ocupacion_diaria").fetchone()[0]`
- Si sale ❌, leed la pista que aparece tras la flecha `→` y corregid.
- Con esto ya tenéis la **serie diaria dentro de la base**: es el primer paso del reto del sprint.

In [ ]:
# 4.1 · A mano: guardar una tabla pequeña de pandas en la base
pequena = pd.DataFrame({"mes": ["01", "02"], "media": [78.8, 74.1]})
pequena.to_sql("tabla_prueba", con, if_exists="replace", index=False)
print("Filas en tabla_prueba:", con.execute("SELECT COUNT(*) FROM tabla_prueba").fetchone()[0])

In [ ]:
# 4.2 · TU TURNO: leer el CSV, guardarlo en la base y contar las filas desde la base
diaria = ...         # pd.read_csv(dato("ocupacion_diaria.csv"))
...                  # diaria.to_sql("ocupacion_diaria", con, if_exists="replace", index=False)
n_dias = ...         # con.execute("SELECT COUNT(*) FROM ocupacion_diaria").fetchone()[0]

In [ ]:
# 4.3 · Comprobación del ejercicio 3 (no hay que cambiar nada)
comprobar("crear: CSV leído", lambda: len(diaria) == 1096, "diaria debe tener 1096 filas: pd.read_csv(dato(\"ocupacion_diaria.csv\")). ¿Has cambiado los puntos ... por tu código?")
comprobar("crear: 1096 filas en la base", lambda: n_dias == 1096, "la tabla de la base debe tener 1096 filas: ¿ejecutaste diaria.to_sql(\"ocupacion_diaria\", con, if_exists=\"replace\", index=False)?")
comprobar("crear: columnas", lambda: [c[1] for c in con.execute("PRAGMA table_info(ocupacion_diaria)").fetchall()] == ["fecha", "ocupacion_pct", "temperatura_c", "festivo", "precio_medio"], "la tabla debe tener 5 columnas: fecha, ocupacion_pct, temperatura_c, festivo y precio_medio. ¿La tabla se llama ocupacion_diaria?")

In [ ]:
# 4.4 · Mirar el resultado: leer 3 filas desde la base
pd.read_sql_query("SELECT * FROM ocupacion_diaria LIMIT 3", con)

## Paso 5 · Cerrar los dos ejercicios (2 min)

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **5.1** | Ejecutad la celda 5.1 (cuenta las comprobaciones ✅ de cada ejercicio por separado). | `Parte A · Leer (ejercicio 1): 4 de 4 ✅`, `Parte B · Crear (ejercicio 3): 3 de 3 ✅` y `🎉 Los ejercicios 1 y 3 están hechos` |
| **5.2** | Si os sobra tiempo, abrid el cuaderno completo [`S05_06_crud_python`](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/S05_06_crud_python.ipynb) y probad los ejercicios 2 (consultas con parámetros) y 4 (actualizar y probar un `rollback`). | — |

In [ ]:
# 5.1 · Resumen por ejercicio
N1 = ["leer: n_reservas", "leer: columnas", "leer: una fila por isla", "leer: tasa de Tenerife"]
N3 = ["crear: CSV leído", "crear: 1096 filas en la base", "crear: columnas"]
bien1 = sum(RESULTADOS.get(n, False) for n in N1)
bien3 = sum(RESULTADOS.get(n, False) for n in N3)
print(f"Parte A · Leer (ejercicio 1): {bien1} de 4 ✅")
print(f"Parte B · Crear (ejercicio 3): {bien3} de 3 ✅")
if bien1 == 4 and bien3 == 3:
    print("🎉 Los ejercicios 1 y 3 están hechos")
else:
    print("Faltan comprobaciones: mirad las líneas ❌ de las celdas 2.3 (ejercicio 1) y 4.3 (ejercicio 3)")

## ✅ Terminado cuando

**Parte A · Leer (ejercicio 1)**

- [ ] La celda 2.3 muestra las cuatro líneas ✅.
- [ ] Cada persona sabe explicar con sus palabras **qué hace `pd.read_sql_query(consulta, con)`**.

**Parte B · Crear (ejercicio 3)**

- [ ] La celda 4.3 muestra las tres líneas ✅.
- [ ] Cada persona sabe explicar con sus palabras **para qué sirve `commit`**.

**Los dos**

- [ ] La celda 5.1 dice `🎉 Los ejercicios 1 y 3 están hechos`.

---

# 2 · Explicaciones

*Para cuando una cosa no sale o queréis entender por qué se hace así. No hace falta leerlo para resolver los ejercicios. Está en dos bloques, igual que la guía: **A · Leer** y **B · Crear**.*

# Explicaciones de la parte A · Leer (ejercicio 1)

## La idea: Python habla con la base de datos

- Una **base de datos** guarda los datos en **tablas**, como hojas de cálculo muy grandes. Se les hacen preguntas en **SQL**.
- Hasta ahora escribíais el SQL en una herramienta aparte. Ahora lo hace **un programa de Python**, y la respuesta llega a Python para seguir trabajando con ella.
- **SQLite** es una base de datos que vive en **un solo fichero** (`turisdata_copia.db`). Python la trae incluida con el módulo `sqlite3`.
- Se trabaja sobre una **copia** para no estropear el fichero original.

## CRUD: las cuatro cosas que se hacen con los datos

| Letra | Significa | En SQL | Hoy |
|---|---|---|---|
| **C** | **C**rear datos nuevos | `INSERT` | Parte B |
| **R** | **R**eer (consultar) | `SELECT` | Parte A |
| **U** | **U**pdate: cambiar datos | `UPDATE` | Cuaderno completo (ejercicio 4) |
| **D** | **D**elete: borrar datos | `DELETE` | Cuaderno completo (ejercicio 4) |

## Cómo se lee de la base: `connect`, `execute` y `fetch`

| Instrucción | Qué hace | Ejemplo |
|---|---|---|
| **`sqlite3.connect(ruta)`** | Abre la conexión con la base (el «cable») | `con = sqlite3.connect(ruta_bd)` |
| **`con.execute("SELECT …")`** | Lanza una pregunta en SQL | `con.execute("SELECT COUNT(*) FROM reservas")` |
| **`.fetchone()`** | Trae **una** fila (como tupla) | `(6000,)` |
| **`.fetchall()`** | Trae **todas** las filas (lista de tuplas) | `[(1, 'Alojamiento 01', 'Tenerife'), …]` |
| **`pd.read_sql_query(sql, con)`** | Lanza la pregunta y devuelve una **tabla de pandas** | La tabla de 1.3 |

- Una **tupla** son valores entre paréntesis. `(6000,)` tiene un solo valor: para sacarlo, `[0]` (el primero): `fetchone()[0]` → `6000`.
- Con pandas la respuesta ya viene con **columnas con nombre**, y se puede analizar o dibujar sin más.

## La consulta del ejercicio 1, línea a línea

```sql
SELECT a.isla,
       ROUND(100.0 * AVG(r.cancelada), 1) AS pct_cancelacion
FROM reservas AS r
JOIN alojamientos AS a ON r.id_alojamiento = a.id_alojamiento
GROUP BY a.isla
ORDER BY a.isla
```

| Línea | Qué hace |
|---|---|
| `FROM reservas AS r` | Parte de la tabla de reservas y la llama `r` (abreviatura) |
| `JOIN alojamientos AS a ON …` | La **une** con la de alojamientos: cada reserva sabe de qué isla es |
| `GROUP BY a.isla` | Agrupa las reservas **por isla** |
| `AVG(r.cancelada)` | La media de unos y ceros = la **fracción** de canceladas (0.46) |
| `100.0 * … , 1` | La pasa a porcentaje y la redondea a 1 decimal (46.0) |
| `AS pct_cancelacion` | Pone **nombre** a esa columna |
| `ORDER BY a.isla` | Ordena las islas alfabéticamente |

## Las palabras de la parte A

| Palabra | Qué significa |
|---|---|
| **Base de datos** | Un conjunto de tablas relacionadas |
| **Tabla** | Filas y columnas, como una hoja de cálculo (`reservas`, `alojamientos`) |
| **SQL** | El lenguaje para preguntar a una base de datos |
| **`con` (conexión)** | El «cable» entre Python y la base |
| **Tupla** | Valores entre paréntesis: `(6000,)` |
| **`JOIN`** | Unir dos tablas por una columna que tienen en común |
| **`GROUP BY`** | Agrupar filas para calcular algo por grupo |
| **Alias (`AS`)** | Un nombre corto o nuevo para una tabla o una columna |
| **`DataFrame`** | La tabla de pandas |
| **✅ / ❌** | Lo que escribe la función `comprobar`: ✅ está bien; ❌ hay algo que corregir (con una pista) |

## Por qué se hace así

- **¿Por qué `fetchone()[0]` y no solo `fetchone()`?**
  - `fetchone()` devuelve una fila entera, aunque tenga un solo valor: `(6000,)`.
  - `[0]` saca el primer elemento: `6000`, un número normal con el que se puede operar.
- **¿Por qué `pd.read_sql_query` y no `fetchall()`?**
  - Con `fetchall()` obtienes tuplas sin nombres de columna. Con pandas obtienes una **tabla con nombres**, lista para analizar.
- **¿Por qué el SQL va en una variable (`consulta`)?**
  - Una consulta larga se lee mejor escrita en varias líneas, entre comillas triples (tres comillas seguidas, al principio y al final), y así se puede reutilizar.
- **¿Por qué copiar la base?**
  - Para poder borrar, cambiar o equivocarse sin riesgo. Si algo sale mal, se vuelve a ejecutar la preparación y listo.

# Explicaciones de la parte B · Crear (ejercicio 3)

## La idea: meter datos en la base

- Hay dos formas de **crear** datos en una base desde Python:

| Forma | Cuándo | Código |
|---|---|---|
| **Una fila** | Añadir un dato suelto | `con.execute("INSERT INTO … VALUES (?, ?)", (a, b))` |
| **Una tabla entera** | Cargar un fichero o una tabla de pandas | `diaria.to_sql("tabla", con, if_exists="replace", index=False)` |

- En el ejercicio 3 usáis la segunda: pasar la tabla de pandas `diaria` a una **tabla de la base**.

## `commit` y `rollback`: guardar y deshacer

- Es como un documento de texto:
  - Lo que escribís **no está guardado** hasta que le dais a «Guardar».
  - Si lo cerráis sin guardar, se pierde.
- En una base de datos:
  - **`con.commit()`** = guardar los cambios de verdad.
  - **`con.rollback()`** = tirar los cambios que aún no se guardaron.
- En el paso 3: la fila 21 se **confirmó** y se queda. La 22 **no se confirmó** y el `rollback` la quitó (se pasó de 22 alojamientos a 21).
- `to_sql` ya **guarda por sí solo** (no hace falta `commit` después).
- Si un programa se interrumpe antes del `commit`, la base queda **como estaba**: así nunca se queda a medias.

## Cómo se lee `to_sql`

`diaria.to_sql("ocupacion_diaria", con, if_exists="replace", index=False)`

| Parte | Qué significa |
|---|---|
| **`diaria`** | La tabla de pandas que queremos guardar |
| **`"ocupacion_diaria"`** | El **nombre** que tendrá la tabla nueva en la base |
| **`con`** | La conexión: en qué base se guarda |
| **`if_exists="replace"`** | Si la tabla ya existe, **se reemplaza** (así podéis ejecutar la celda varias veces sin error) |
| **`index=False`** | No guardar la numeración 0, 1, 2… de pandas como columna extra |

- Después se comprueba **leyendo desde la base** (`SELECT COUNT(*)`): así se sabe que los datos entraron de verdad y no solo que existen en pandas.
- Una tabla de la base permite luego usar **SQL** sobre la serie diaria (promedios por mes, etc.), como en el reto.

## Las palabras de la parte B

| Palabra | Qué significa |
|---|---|
| **`INSERT`** | Añadir una fila a una tabla |
| **Marcador `?`** | Un hueco en el SQL que se rellena con un valor de la tupla; evita errores y ataques |
| **`commit`** | Confirmar (guardar) los cambios |
| **`rollback`** | Deshacer los cambios no confirmados |
| **`to_sql`** | Guardar una tabla de pandas como tabla de la base |
| **`if_exists="replace"`** | Si la tabla existe, sustituirla |
| **`PRAGMA table_info`** | Una orden de SQLite que lista las columnas de una tabla |
| **CSV** | Un fichero de texto con una tabla: una fila por línea, valores separados por comas |

## Por qué se hace así

- **¿Por qué se ponen los valores aparte, con `?`?**
  - Si se pegara el texto directamente dentro del SQL, un usuario malintencionado podría escribir **código SQL** en un campo. Se llama *inyección SQL*.
  - Con `?` la base trata el valor siempre como un dato, nunca como una orden.
- **¿Por qué guardar el CSV en una base si ya se puede leer con pandas?**
  - Una base es **un solo fichero** con todas las tablas, se puede consultar con SQL y varias personas pueden usarla.
  - Es lo que pide el **reto**: cargar los datos en una base propia.
- **¿Por qué `if_exists="replace"`?**
  - Sin él, volver a ejecutar la celda daría un error porque la tabla ya existe.
- **¿Por qué comprobar con `COUNT(*)` desde la base?**
  - Es la prueba de que el dato **está dentro**, no solo en la memoria de Python.

## 🚑 Si os atascáis

**Parte A · Leer (ejercicio 1)**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `NameError: name 'con' is not defined` | No se ha ejecutado la celda 1.1 | Ejecutad las celdas desde el paso 0 hasta la 1.1, en orden |
| `❌ leer: n_reservas` con el valor `(6000,)` | Falta el `[0]` del final | `…fetchone()[0]` |
| `❌ leer: columnas` | La tabla no sale de la consulta | `pd.read_sql_query(consulta, con)`, con la `consulta` de la celda 2.2 |
| `OperationalError: no such table: …` | Un nombre de tabla mal escrito | Las tablas son `reservas` y `alojamientos`, en minúsculas |
| `OperationalError: near …: syntax error` | La consulta se ha modificado y tiene un fallo | Dejadla tal cual estaba |
| `TypeError` o `AttributeError` que nombran `'ellipsis'` | Se ha quedado algún `...` sin sustituir | Cambiad los `...` por vuestro código y volved a ejecutar 2.2 |
| `ProgrammingError: Cannot operate on a closed database` | La conexión se cerró | Ejecutad otra vez la celda 1.1 |

**Parte B · Crear (ejercicio 3)**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `NameError: name 'con' is not defined` en 3.1 | No se ejecutó la celda 1.1 | Volved al paso 1 y ejecutad la celda 1.1 |
| En 3.1 aparece la fila 21 **repetida** (o en 3.2 salen 23 y 22) | La celda 3.1 se ha ejecutado dos veces | No pasa nada grave: la base es una copia. Para empezar de cero, ejecutad la celda 0.2 y la 1.1 |
| `❌ crear: CSV leído` | `diaria` no tiene los 1096 días | `diaria = pd.read_csv(dato("ocupacion_diaria.csv"))` |
| `❌ crear: 1096 filas en la base` | No se ha guardado la tabla, o falta `n_dias` | Ejecutad la línea del `to_sql` y después la de `n_dias` |
| `OperationalError: no such table: ocupacion_diaria` | El nombre no coincide | Debe ser `"ocupacion_diaria"`, tal cual, entre comillas |
| `❌ crear: columnas` | La tabla tiene una columna extra `index` | Añadid `index=False` al `to_sql` |
| `database is locked` | Hay otra conexión abierta | Reiniciad el entorno (`Entorno de ejecución` → `Reiniciar`) y ejecutad desde el paso 0 |

**En cualquier parte**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `NameError: name 'pd' is not defined` (o `sqlite3`…) | No habéis ejecutado la celda 0.2, o Colab se reinició | Ejecutad las celdas **desde el paso 0** hasta la que falla, en orden |
| La comprobación da ❌ aunque creéis que está bien | Habéis corregido el código pero no habéis vuelto a ejecutar la celda del ejercicio | Ejecutad la celda del ejercicio **otra vez** y después la de comprobación |
| Sale un error al descargar el fichero (`URLError`, `FileNotFoundError`) | Sin internet o sin acceso a GitHub | Probad otra vez; si sigue, avisad al docente (puede dar los ficheros `turisdata.db` y `ocupacion_diaria.csv`) |
| Seguís sin verlo | A veces pasa | 1) Preguntad a vuestra pareja; 2) después, al docente; 3) **cuando hayáis terminado o agotado las pistas**, ejecutad la «Solución de referencia» del ejercicio. Primero intentadlo: lo que copiéis, tenéis que saber explicarlo |

## 🆘 Soluciones de referencia

- Ejecutad estas celdas **solo después de intentarlo**.
- Cada ejercicio tiene la suya.
- Si lo dejáis para después, ejecutadlas para poder seguir con el paso siguiente.

In [ ]:
#@title 🆘 Solución del ejercicio 1 · Leer (ábrela solo después de intentarlo)
n_reservas = con.execute("SELECT COUNT(*) FROM reservas").fetchone()[0]
tasa_por_isla = pd.read_sql_query(consulta, con)
print("Solución del ejercicio 1 aplicada: ahora ejecutad la celda 2.3")

In [ ]:
#@title 🆘 Solución del ejercicio 3 · Crear (ábrela solo después de intentarlo)
diaria = pd.read_csv(dato("ocupacion_diaria.csv"))
diaria.to_sql("ocupacion_diaria", con, if_exists="replace", index=False)
n_dias = con.execute("SELECT COUNT(*) FROM ocupacion_diaria").fetchone()[0]
print("Solución del ejercicio 3 aplicada: ahora ejecutad la celda 4.3")

## Para reflexionar (con vuestra pareja)

**Parte A · Leer**

1. ¿Qué diferencia hay entre recibir la respuesta con `fetchall()` y con `pd.read_sql_query`? ¿Cuándo usaríais cada una?
2. La tasa de cancelación de cada isla ronda el 46 %. ¿Qué os dice eso sobre si la isla influye en las cancelaciones?

**Parte B · Crear**

1. ¿Qué pasaría con la fila 21 si el programa se interrumpiera justo **antes** del `commit`?
2. ¿Por qué creéis que el reto pide guardar los datos en una base en lugar de leer siempre el CSV?

## Qué habéis aprendido

**Parte A · Leer**

- **CRUD** = Crear, Leer, Actualizar y Borrar; hoy habéis hecho la **R** y la **C**.
- `sqlite3.connect` abre la conexión; `execute` + `fetchone` / `fetchall` traen los datos.
- `pd.read_sql_query(consulta, con)` devuelve la respuesta como una tabla de pandas.

**Parte B · Crear**

- `INSERT` con `?` añade una fila; los cambios no son definitivos hasta `commit`, y `rollback` los deshace.
- `to_sql` guarda una tabla de pandas como tabla de la base.
- Ya tenéis la serie diaria **dentro de la base**: es el paso 1 del reto.